# MFVI predictive-variance — full reproducibility gate

Use a **CPU** Colab runtime. This notebook does not create a virtual environment. It selects a usable Colab Python interpreter and passes it explicitly to the complete nine-dataset source run, independent audit, and fail-closed publication gate.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

uploaded = files.upload()
archive = next((Path(name) for name in uploaded if name.endswith('.tar.gz')), None)
if archive is None:
    raise RuntimeError('Upload RG7maF4bGu-colab-bundle.tar.gz')
subprocess.run(['tar', '-tzf', str(archive)], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['tar', '-xzf', str(archive), '-C', '/content'], check=True)
PROJECT = Path('/content/icml26-repro-RG7maF4bGu-mfvi-predictive-variance')
if not PROJECT.is_dir():
    raise RuntimeError(f'Expected project directory missing: {PROJECT}')
os.chdir(PROJECT)
print('Input bundle extracted to:', PROJECT)

In [ ]:
# Select an interpreter that actually imports every dependency. No venv is used.
def find_usable_python():
    candidates = []
    for candidate in (sys.executable, shutil.which('python'), shutil.which('python3')):
        if candidate and candidate not in candidates:
            candidates.append(candidate)
    diagnostics = {}
    probe = 'import torch, pandas, numpy, scipy, pytest; print(torch.__version__)'
    for candidate in candidates:
        result = subprocess.run([candidate, '-c', probe], text=True, capture_output=True)
        diagnostics[candidate] = (result.returncode, result.stdout, result.stderr)
        if result.returncode == 0:
            return candidate, diagnostics
    pip = shutil.which('pip') or shutil.which('pip3')
    if pip is not None:
        subprocess.run([pip, 'install', '--quiet', '-r', 'repro/requirements.txt'], check=True)
        subprocess.run([pip, 'install', '--quiet', 'torch'], check=True)
        for candidate in candidates:
            result = subprocess.run([candidate, '-c', probe], text=True, capture_output=True)
            diagnostics[candidate] = (result.returncode, result.stdout, result.stderr)
            if result.returncode == 0:
                return candidate, diagnostics
    raise RuntimeError('No usable Python interpreter after dependency check: ' + repr(diagnostics))

PYTHON_BIN, diagnostics = find_usable_python()
subprocess.run([PYTHON_BIN, '-c', 'import torch, pandas, numpy, scipy, pytest; print({\"python\": __import__(\"sys\").executable, \"torch\": torch.__version__, \"cuda\": torch.cuda.is_available()})'], check=True)
print('Selected interpreter:', PYTHON_BIN)

In [ ]:
# Trust only this extracted pinned Git checkout, then run the complete CPU gate.
subprocess.run(['git', 'config', '--global', '--add', 'safe.directory', str(PROJECT / 'upstream')], check=True)
environment = {**os.environ, 'PYTHON_BIN': PYTHON_BIN, 'CUDA_VISIBLE_DEVICES': '', 'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
subprocess.run([PYTHON_BIN, 'repro/src/verify_mfvi.py', '--mode', 'synthetic', '--output', 'outputs/colab_synthetic_preflight.json'], check=True, env=environment)
subprocess.run(['bash', 'repro/src/run_full_gate.sh'], check=True, env=environment)
subprocess.run([PYTHON_BIN, '-m', 'pytest', '-q', 'repro/tests'], check=True, env=environment)
manifest = json.loads((PROJECT / 'outputs/prepublish_gate.json').read_text())
assert manifest['publication_gate_passed'] is True
print(json.dumps(manifest, indent=2, sort_keys=True))

In [ ]:
# Download the outputs-only evidence archive and upload that archive back to this chat.
result_archive = Path('/content/RG7maF4bGu-colab-results.tar.gz')
subprocess.run(['tar', '-czf', str(result_archive), '-C', str(PROJECT), 'outputs'], check=True)
subprocess.run(['sha256sum', str(result_archive)], check=True)
files.download(str(result_archive))